## Obtain corners from calib images

In [51]:
import numpy as np
import cv2
import glob

In [52]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
criteria

(3, 30, 0.001)

In [53]:
objp = np.zeros((6*9,3), np.float32)
objp[:,:2] = np.mgrid[0:9,0:6].T.reshape(-1,2)
objp *= 24


In [54]:
objpoints = []
imgpoints = []

In [55]:
images = glob.glob('./image/*.jpeg')
images

['./image\\calib_img_0.jpeg',
 './image\\calib_img_120.jpeg',
 './image\\calib_img_150.jpeg',
 './image\\calib_img_180.jpeg',
 './image\\calib_img_210.jpeg',
 './image\\calib_img_240.jpeg',
 './image\\calib_img_270.jpeg',
 './image\\calib_img_30.jpeg',
 './image\\calib_img_300.jpeg',
 './image\\calib_img_330.jpeg',
 './image\\calib_img_360.jpeg',
 './image\\calib_img_390.jpeg',
 './image\\calib_img_420.jpeg',
 './image\\calib_img_450.jpeg',
 './image\\calib_img_480.jpeg',
 './image\\calib_img_510.jpeg',
 './image\\calib_img_540.jpeg',
 './image\\calib_img_570.jpeg',
 './image\\calib_img_60.jpeg',
 './image\\calib_img_600.jpeg',
 './image\\calib_img_630.jpeg',
 './image\\calib_img_660.jpeg',
 './image\\calib_img_690.jpeg',
 './image\\calib_img_720.jpeg',
 './image\\calib_img_750.jpeg',
 './image\\calib_img_780.jpeg',
 './image\\calib_img_810.jpeg',
 './image\\calib_img_840.jpeg',
 './image\\calib_img_90.jpeg']

In [56]:
for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    ret, corners = cv2.findChessboardCorners(gray, (9,6), None)

    if ret:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)

        cv2.drawChessboardCorners(img, (9,6), corners2, ret)
        
        cv2.imwrite(fname.split('\\')[-1], img)
        
        cv2.imshow('img', img)
        cv2.waitKey(0)
cv2.destroyAllWindows()

In [57]:
ret, mtx, dist, rvec, tvec = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

In [59]:
for fname in images:
    img = cv2.imread(fname)
    # h, w = img.shape[:2]
    # newcameramtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 1, (w,h))
    dst = cv2.undistort(img, mtx, dist, None) #, newcameramtx
    #x, y, w, h = roi
    #dst = dst[y:y+h, x:x+w]
    cv2.imshow('calibrated result', dst)
    name = fname.split('\\')[-1]
    cv2.imwrite(f"./image_calibrated/{name}",dst)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [64]:
mtx

array([[1.93323040e+03, 0.00000000e+00, 8.08499872e+02],
       [0.00000000e+00, 1.92921965e+03, 6.92750958e+01],
       [0.00000000e+00, 0.00000000e+00, 1.00000000e+00]])

In [61]:
np.save('./calibration.npy', {'mtx': mtx, 'dist':dist})

Obtain focal length using focal len *px size (sensor size in mm/num of pixels)    

In [62]:
px_size = 0.00345 #in mm
mtx[0,0] * px_size , mtx[1,1] * px_size

(6.669644888238692, 6.655807782590527)

Re-projection error

In [63]:
mean_error = 0
for i in range(len(objpoints)):
    imgpoints2, _ = cv2.projectPoints(objpoints[i], rvec[i], tvec[i], mtx, dist)
    error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2)/len(imgpoints2)
    mean_error+=error
print("total error: {}".format(mean_error/len(objpoints  )))

total error: 0.039604283811485695
